# Supplementary tables

### Setup

In [ ]:
%run setup.ipynb
%matplotlib inline

In [ ]:
# load haplotypes
callset_haps = np.load('../data/haps_phase1.npz')
haps = allel.HaplotypeArray(callset_haps['haplotypes'])
pos = allel.SortedIndex(callset_haps['POS'])
n_variants = haps.shape[0]
n_haps = haps.shape[1]
n_variants, n_haps

In [ ]:
list(callset_haps)

In [ ]:
callset_haps['ANN']

In [ ]:
callset_phased = phase1_ar31.callset_phased
sorted(callset_phased['2L/variants'])

In [ ]:
# load up haplotype group assignments from hierarchical clustering
hierarchical_group_membership = np.load('../data/hierarchical_cluster_membership.npy')
np.unique(hierarchical_group_membership)

In [ ]:
# load up haplotype group assignments from network analysis
network_group_membership = np.load('../data/median_joining_network_membership.npy')
network_group_membership[network_group_membership == ''] = 'WT'
np.unique(network_group_membership)

In [ ]:
# load up core haplotypes
with open('../data/core_haps.pkl', mode='rb') as f:
    core_haps = pickle.load(f)

## Table of non-synonymous variants

One row per alternate allele.

In [ ]:
tbl_variants = etl.frompickle('../data/tbl_variants_phase1.pkl')
tbl_variants.head()

In [ ]:
transcript_ids = [
    'AGAP004707-RA',
    'AGAP004707-RB',
    'AGAP004707-RC',
    'Davies-C1N2',
    'Davies-C3N2',
    'Davies-C5N2',
    'Davies-C7N2',
    'Davies-C8N2',
    'Davies-C10N2',
    'Davies-C11N2',
    'Davies-C1N9',
    'Davies-C8N9',
    'Davies-C1N9ck'
]

In [ ]:
#load the codon map from the blog post (with the header info removed)
md_tbl = etl.fromtsv('../data/domestica_gambiae_map.txt')
md_tbl

In [ ]:
dom_fs = pyfasta.Fasta('../data/domestica_gambiae_PROT_MEGA.fas')

#grab the right sample
dom = dom_fs.get('domestica_vgsc')

#remove the '-' from the aligned fasta so the numbering makes sense
dom_fix = [p for p in dom if p != '-']
#check
dom_fix[261-1],dom_fix[1945-1]


In [ ]:
# keep only the missense variants


def simplify_missense_effect(v):
    if v and v[0] == 'NON_SYNONYMOUS_CODING':
        return v[1]
    else:
        return ''
    
    
tbl_variants_missense = (
    tbl_variants
    .select(lambda row: any(row[t] and row[t][0] == 'NON_SYNONYMOUS_CODING' for t in transcript_ids))
    .convert(transcript_ids, simplify_missense_effect)
    .capture('AGAP004707-RA', pattern="([0-9]+)", newfields=['gambiae_codon'], include_original=True, fill=[''])
    .hashleftjoin(md_tbl, key='gambiae_codon', missing='')
    .replace('domestica_codon', '.', '')
    .convert('domestica_codon', lambda v: dom_fix[int(v) - 1] + v if v else v)
    .cut('CHROM', 'POS', 'REF', 'ALT', 'AC', 'exon', 'domestica_codon',
         'AGAP004707-RA', 'AGAP004707-RB', 'AGAP004707-RC', 'Davies-C1N2', 'Davies-C3N2', 'Davies-C5N2', 
         'Davies-C7N2', 'Davies-C8N2', 'Davies-C10N2', 'Davies-C11N2', 'Davies-C1N9', 'Davies-C8N9', 'Davies-C1N9ck',
         'AF_AOM', 'AF_BFM', 'AF_GWA', 'AF_GNS', 'AF_BFS', 'AF_CMS', 'AF_GAS', 'AF_UGS', 'AF_KES',
         'FILTER_PASS', 'NoCoverage', 'LowCoverage', 'HighCoverage', 'LowMQ', 'HighMQ0', 'RepeatDUST', 'RepeatMasker', 'RepeatTRF', 'FS', 'HRun', 'QD', 'ReadPosRankSum',
         )
    

)
tbl_variants_missense.displayall()

In [ ]:
tbl_variants_missense.nrows()

In [ ]:
tbl_variants_missense.tocsv('../data/supp_table_variants_missense.csv')

## Table of haplotype tracking variants

One row per variant. Only biallelic obviously because haplotype tagging.

Columns:
* CHROM
* POS
* REF
* ALT
* AC
* AF
* AF_F1
* AF_F2
* AF_F3
* AF_F4
* AF_F5
* AF_S1
* AF_S2
* AF_S3
* AF_S4
* AF_S5
* AF_L1
* AF_L2
* AF_WT


In [ ]:
list(callset_phased['2L/variants'])

In [ ]:
region_vgsc

In [ ]:
pos = allel.SortedIndex(callset_phased['2L/variants/POS'])
pos

In [ ]:
# "_tr" means tracking region, i.e., genome region we'll report data for tracking SNPs
loc_tr = pos.locate_range(region_vgsc.start - 10000, region_vgsc.end + 10000)
loc_tr

In [ ]:
pos_tr = pos[loc_tr]
pos_tr

In [ ]:
haps_tr = allel.GenotypeArray(callset_phased['2L/calldata/genotype'][loc_tr, :-8]).to_haplotypes()
haps_tr

In [ ]:
ac_tr = haps_tr.count_alleles(max_allele=1)
ac_tr

In [ ]:
af_tr = ac_tr.to_frequencies()
af_tr

In [ ]:
subpops = dict()
for p in 'F1 F2 F3 F4 F5 S1 S2 S3 S4 S5 L1 L2 WT'.split():
    subpops[p] = np.nonzero(network_group_membership == p)[0]
sorted((k, len(v)) for k, v in subpops.items())

In [ ]:
ac_subpops_tr = haps_tr.count_alleles_subpops(subpops, max_allele=1)

In [ ]:
af_subpops_tr = {k: ac.to_frequencies()[:, 1] for k, ac in ac_subpops_tr.items()}

In [ ]:
columns = [
    ('CHROM', np.asarray(['2L'] * haps_tr.shape[0], dtype=object)),
    ('POS', callset_phased['2L/variants/POS'][loc_tr]),
    ('REF', callset_phased['2L/variants/REF'][loc_tr].astype('U')),
    ('ALT', callset_phased['2L/variants/ALT'][loc_tr].astype('U')),
    ('AC', ac_tr[:, 1]),
    ('AF', af_tr[:, 1]),
    ('MAC', ac_tr.min(axis=1)),
    ('MAF', af_tr.min(axis=1)),
] + sorted(('AF_' + k, v) for k, v in af_subpops_tr.items())
df_tr = pandas.DataFrame.from_items(columns)
df_tr.head()

In [ ]:
df_tr_mac = df_tr[df_tr.MAC > 14].reset_index(drop=True)
df_tr_mac.head()

In [ ]:
# check how many potentially diagnostics SNPs separating each pair of F haplotype groups
for x, y in itertools.combinations('F1 F2 F3 F4 F5'.split(), 2):
    print(x, y, np.count_nonzero(np.abs(df_tr_mac['AF_' + x] - df_tr_mac['AF_' + y]) > 0.98))

In [ ]:
# check how many potentially diagnostics SNPs separating each pair of S haplotype groups
for x, y in itertools.combinations('S1 S2 S3 S4 S5'.split(), 2):
    print(x, y, np.count_nonzero(np.abs(df_tr_mac['AF_' + x] - df_tr_mac['AF_' + y]) > 0.98))

In [ ]:
df_tr_mac.to_csv('../data/supp_table_variants_tracking.csv', index=False, float_format='%.3f')

## Table of haplotypes panel

In [ ]:
df_haps = pandas.read_csv('../data/ag1000g.phase1.AR3.1.haplotypes.meta.txt', sep='\t', index_col='index')[:-16]
df_haps.head()

In [ ]:
n_haps

In [ ]:
core_hap_col = np.empty(n_haps, dtype=object)
for k, s in core_haps.items():
    core_hap_col[sorted(s)] = k
core_hap_col

In [ ]:
df_haps_out = (
    df_haps[['label', 'ox_code', 'population', 'country', 'region', 'sex']]
    .rename(columns={'label': 'haplotype_id', 'ox_code': 'sample_id'})
    .copy()
)
df_haps_out['core_haplotype'] = core_hap_col
df_haps_out['network_haplotype_group'] = network_group_membership
df_haps_out['hierarchy_haplotype_group'] = hierarchical_group_membership.astype('U')
df_haps_out

In [ ]:
df_haps_out[df_haps_out.core_haplotype == 'L1']

In [ ]:
df_haps_out[df_haps_out.core_haplotype == 'L2']

In [ ]:
pandas.set_option('display.max_rows', 9999)
df_haps_out.groupby(by=('population', 'network_haplotype_group')).count()[['haplotype_id']]

In [ ]:
df_haps_out.to_csv('../data/supp_table_haplotype_panel.csv', index=False)

## Table of haplotype data

In [ ]:
haps_tr_mac = haps_tr[df_tr.MAC > 14]
haps_tr_mac

In [ ]:
df_hap_data = df_tr_mac[['CHROM', 'POS', 'REF', 'ALT']].merge(
    pandas.DataFrame(np.asarray(haps_tr_mac), columns=df_haps.label), 
    left_index=True, right_index=True)
df_hap_data.head()

In [ ]:
df_hap_data.to_csv('../data/supp_table_haplotypes_tracking.csv', index=False)

## All variation, for PCR flanks

In [ ]:
callset = phase1_ar31.callset
callset

In [ ]:
pos = allel.SortedIndex(callset['2L/variants/POS'])
pos

In [ ]:
loc_cs_tr = pos.locate_range(region_vgsc.start - 10000, region_vgsc.end + 10000)
pos_cs_tr = pos[loc_cs_tr]
pos_cs_tr

In [ ]:
gt = allel.GenotypeArray(callset['2L/calldata/genotype'][loc_cs_tr])
gt

In [ ]:
df_samples = phase1_ar3.df_samples
df_samples.head()

In [ ]:
subpops = {p: sorted(df_samples[df_samples.population == p].index.values)
           for p in df_samples.population.unique()}
acs = gt.count_alleles_subpops(subpops, max_allele=3)
acs['BFS']

In [ ]:
afs = {p: acs[p].to_frequencies()[:, 1:].sum(axis=1) for p in df_samples.population.unique()}
afs

In [ ]:
columns = ([
    ('CHROM', callset['2L/variants/CHROM'][loc_cs_tr].astype('U')),
    ('POS', callset['2L/variants/POS'][loc_cs_tr]),
    ('REF', callset['2L/variants/REF'][loc_cs_tr].astype('U')),
    ('ALT_1', callset['2L/variants/ALT'][loc_cs_tr, 0].astype('U')),
    ('ALT_2', callset['2L/variants/ALT'][loc_cs_tr, 1].astype('U')),
    ('ALT_3', callset['2L/variants/ALT'][loc_cs_tr, 2].astype('U')),
    ('AC_1', callset['2L/variants/AC'][loc_cs_tr, 0]),
    ('AC_2', callset['2L/variants/AC'][loc_cs_tr, 1]),
    ('AC_3', callset['2L/variants/AC'][loc_cs_tr, 2]),
    ('AF_1', callset['2L/variants/AF'][loc_cs_tr, 0]),
    ('AF_2', callset['2L/variants/AF'][loc_cs_tr, 1]),
    ('AF_3', callset['2L/variants/AF'][loc_cs_tr, 2]),
    ('AF_total', callset['2L/variants/AF'][loc_cs_tr, :].sum(axis=1)),
    ('AN', callset['2L/variants/AN'][loc_cs_tr]),
    ('FILTER_PASS', callset['2L/variants/FILTER_PASS'][loc_cs_tr]),
] + [('AF_' + p, afs[p]) for p in df_samples.population.unique()] +
    [('MAX_POP_AF', np.column_stack([afs[p] for p in df_samples.population.unique()]).max(axis=1))]
)
df_cs_tr = pandas.DataFrame.from_items(columns)
df_cs_tr.head()

In [ ]:
df_cs_tr.FILTER_PASS.describe()

In [ ]:
df_cs_tr.to_csv('../data/supp_table_variants_all.csv')